# IMDb-faces — exploratory data analysis

Cropped face photographs of IMDb personalities, labelled by gender (binary classification, scored on accuracy) and tagged with the year each photo was taken. The dataset is used to study temporal drift across decades of photography.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl

from drift_happens.dataset.imdb_faces.load import load_preprocessed_df

plt.rcParams["axes.unicode_minus"] = False
PRIMARY, SECONDARY = "#4C78A8", "#E45756"

imdb = load_preprocessed_df()
imdb.select(["sample_idx", "photo_taken", "gender", "age"]).head()

## Dataset at a glance

In [ ]:
key_cols = ["gender", "photo_taken", "age"]
missing = max(imdb[c].null_count() / imdb.height for c in key_cols)
pd.Series({
    "Samples": f"{imdb.height:,}",
    "Years": f"{imdb['photo_taken'].min()}–{imdb['photo_taken'].max()}",
    "Classes": "2 (female, male)",
    "Missing (gender, photo_taken, age)": f"{missing:.1%}",
    "Task": "binary classification",
    "Metric": "accuracy",
}, name="imdb_faces")

## Temporal coverage

In [ ]:
per_year = imdb.group_by("photo_taken").len().sort("photo_taken")

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(per_year["photo_taken"], per_year["len"], color=PRIMARY)
ax.set_xlabel("Photo taken (year)")
ax.set_ylabel("Faces")
ax.set_title("Faces per year")
fig.tight_layout()

## Gender balance

Gender is encoded as 0 (female), 1 (male); missing values are unlabelled.

In [ ]:
names = {0: "Female", 1: "Male"}
counts = imdb["gender"].value_counts().sort("gender")
labels = [names.get(g, "Unlabelled") for g in counts["gender"].to_list()]

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(labels, counts["count"].to_list(), color=PRIMARY)
ax.set_ylabel("Faces")
ax.set_title("Gender distribution")
fig.tight_layout()

## Unlabelled gender

In [ ]:
unlabelled = imdb["gender"].null_count()
print(f"{unlabelled:,} of {imdb.height:,} faces ({unlabelled / imdb.height:.1%}) have no gender label.")

## Gender balance over time

Excluding unlabelled faces, the share of each gender shifts across decades.

In [ ]:
gender_year = (
    imdb.filter(pl.col("gender").is_not_null())
    .group_by(["photo_taken", "gender"])
    .len()
    .to_pandas()
    .pivot(index="photo_taken", columns="gender", values="len")
    .fillna(0)
    .sort_index()
)
share = gender_year.div(gender_year.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(10, 4))
ax.stackplot(share.index, share[0], share[1], labels=["Female", "Male"],
             colors=[SECONDARY, PRIMARY])
ax.set_xlabel("Photo taken (year)")
ax.set_ylabel("Share")
ax.set_ylim(0, 1)
ax.set_title("Gender share per year")
ax.legend(loc="lower right")
fig.tight_layout()

## Age over time

In [ ]:
age_year = (
    imdb.filter((pl.col("age") >= 0) & (pl.col("age") <= 100))
    .group_by("photo_taken")
    .agg(pl.col("age").median().alias("median_age"))
    .sort("photo_taken")
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(age_year["photo_taken"], age_year["median_age"], color=PRIMARY)
ax.set_xlabel("Photo taken (year)")
ax.set_ylabel("Median age")
ax.set_title("Median age per year")
fig.tight_layout()

## Age distribution

In [ ]:
ages = imdb["age"].drop_nulls().to_numpy()
ages = ages[(ages >= 0) & (ages <= 100)]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(ages, bins=50, color=PRIMARY)
ax.set_xlabel("Age at photo")
ax.set_ylabel("Faces")
ax.set_title("Age distribution")
fig.tight_layout()

## Sample faces across the decades

In [ ]:
names = {0: "F", 1: "M"}
by_decade = (
    imdb.filter(pl.col("photo_taken").is_not_null())
    .with_columns((pl.col("photo_taken") // 10 * 10).alias("decade"))
)
sample = (
    by_decade.sort(["decade", "sample_idx"])
    .unique(subset="decade", keep="first", maintain_order=True)
    .sort("decade")
)
decades = sample["decade"].to_list()
ncols = 6
nrows = math.ceil(len(decades) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(12, 2.3 * nrows))
axes = np.atleast_1d(axes).ravel()
for ax, img, gender, year in zip(axes, sample["img"].to_numpy(), sample["gender"], sample["photo_taken"]):
    face = np.asarray(img, dtype=float)
    ax.imshow(face / 255 if face.max() > 1 else face)
    ax.set_title(f"{year} · {names.get(gender, '?')}", fontsize=9)
    ax.axis("off")
for ax in axes[len(decades):]:
    ax.axis("off")
fig.suptitle("One face per decade")
fig.tight_layout()

## Mean face per decade

Averaging every face in a decade exposes drift in photo quality, colour, and framing.

In [ ]:
def mean_image(images):
    stacked = np.stack([np.asarray(img, dtype=float) for img in images]).mean(axis=0)
    return stacked / 255 if stacked.max() > 1 else stacked

decades = sorted(by_decade["decade"].unique().to_list())
ncols = 6
nrows = math.ceil(len(decades) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(12, 2.3 * nrows))
axes = np.atleast_1d(axes).ravel()
for ax, decade in zip(axes, decades):
    images = by_decade.filter(pl.col("decade") == decade)["img"].to_list()
    ax.imshow(mean_image(images))
    ax.set_title(f"{decade}s · n={len(images)}", fontsize=8)
    ax.axis("off")
for ax in axes[len(decades):]:
    ax.axis("off")
fig.suptitle("Mean face per decade")
fig.tight_layout()